# 10 · Hybrid Search (Reciprocal Rank Fusion)

**Idea:** run dense **and** BM25, then fuse their ranked lists. **RRF** gives each hit a score of `1 / (rrf_k + rank)` in each list and sums them — no score calibration needed between the two systems. The result beats either retriever alone on mixed queries.

> Needs Ollama running (for the dense half) and `rank-bm25`.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, NumpyStore, BM25Store, HybridRetriever

corpus = load_markdown()
dense = NumpyStore(provider='ollama'); dense.add(corpus)
sparse = BM25Store(); sparse.add(corpus)
hybrid = HybridRetriever(dense, sparse)
print('dense + sparse over', len(corpus), 'chunks')

A query with **both** an exact token (`RRF`) and a concept ("combine retrievers"). Compare all three retrievers.

In [ ]:
query = 'how does RRF combine two retrievers?'

def show(label, hits):
    print(label)
    for h in hits:
        print(f'  {h.score:6.3f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

show('Dense:', dense.search(query, k=3))
show('BM25:', sparse.search(query, k=3))
show('Hybrid (RRF):', hybrid.search(query, k=3))

Notice the hybrid scores are small `1/(rrf_k+rank)` sums — their *magnitude* is meaningless, only the *order* matters. A chunk ranked highly by **both** retrievers rises to the top even if neither ranked it #1.

In [ ]:
# Tighten fusion by lowering rrf_k (ranks matter more); widen the candidate pool.
tuned = HybridRetriever(dense, sparse, rrf_k=10)
show('Hybrid (rrf_k=10, pool=20):', tuned.search(query, k=3, pool=20))

**Takeaway:** hybrid retrieval is the reliable default in production RAG — it recovers both exact-token and paraphrase matches with a fusion rule that needs no tuning to work. Reranking (next) then sharpens the final order.

Next → `11_reranking.ipynb`